In [28]:
import os, sys
import pandas as pd
import numpy as np

cwd = os.getcwd()

root_path = cwd
while os.path.basename(root_path) != 'PhD_article_1':
    root_path =  os.path.dirname(root_path)
FOB_raw_path = os.path.join(root_path,'data','raw','FOB')
display(os.listdir(FOB_raw_path))

['ReadMe', 'XPAR_SHARES_F1_20231027.csv', 'XPAR_SHARES_F1_20231027.csv.zip']

In [2]:
FOB_files = [f for f in os.listdir(FOB_raw_path) if os.path.splitext(f)[1] == '.csv']
display(FOB_files)

['XPAR_SHARES_F1_20231027.csv']

In [3]:
chunks = []
for chunk in pd.read_csv(os.path.join(FOB_raw_path, FOB_files[0]), 
                        header=0, 
                        low_memory=False, 
                        chunksize=10000, 
                        usecols=['isin',
                                'event_date',
                                'event_time_cet',
                                'order_id',
                                'order_event_type',
                                'order_side',
                                'order_price',
                                'order_size',
                                'order_type',
                                'time_in_force', 
                                'trade_size', 
                                'trade_price']):
    chunk = chunk[(chunk['isin'] == 'FR0000120172')]

    chunk['event_time_cet'] = pd.to_datetime(chunk['event_date'] + ' ' + chunk['event_time_cet'])
    chunk = chunk.drop(columns=['event_date'])
    chunks.append(chunk)

FOB = pd.concat(chunks)
FOB['time_in_force'] = FOB['time_in_force'].astype(str)

In [28]:
CO = FOB.copy()
ls_order = CO.loc[CO['order_event_type'] == 'Cancel', 'order_id']
CO = CO.loc[CO['order_id'].isin(ls_order), ['order_id', 'order_event_type', 'event_time_cet', 'order_side', 'order_size']]

def attrib_last_size(chunk):
    chunk.loc[chunk['order_event_type'] == 'Cancel', 'order_size'] = chunk.sort_values('event_time_cet').loc[chunk['order_event_type'] != 'Cancel', 'order_size'].iloc[-1]
    return chunk.loc[chunk['order_event_type'] == 'Cancel']

CO = CO.groupby(['order_id'], as_index=False).apply(attrib_last_size)
CO.set_index('event_time_cet', inplace=True)
CO = CO.groupby(['order_side']).resample('5min').sum()['order_size'].to_frame()
CO = CO.reset_index().groupby(['event_time_cet', 'order_side'], as_index=False).last()
CO = CO[CO['order_size'] != 0].set_index('event_time_cet')

In [6]:
display(FOB)

,isin,event_time_cet,order_id,order_event_type,order_side,order_price,order_size,order_type,time_in_force,trade_size,trade_price,previous_price,previous_size
initial_index,,,,,,,,,,,,,
4063404,FR0000120172,2023-10-27 03:01:12.295196555,50416736,Reload,Sell,18.60,254.0,Limit,0,0.0,0.0,NaN,NaN
4063405,FR0000120172,2023-10-27 03:01:12.295196555,687950912,Reload,Sell,20.00,26.0,Limit,0,0.0,0.0,NaN,NaN
4063406,FR0000120172,2023-10-27 03:01:12.295196555,755059776,Reload,Sell,19.00,20.0,Limit,0,0.0,0.0,NaN,NaN
4063407,FR0000120172,2023-10-27 03:01:12.295196555,771837112,Reload,Sell,20.00,113.0,Limit,0,0.0,0.0,NaN,NaN
4063408,FR0000120172,2023-10-27 03:01:12.295196555,1174490178,Reload,Buy,15.00,200.0,Limit,0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4336684,FR0000120172,2023-10-27 17:40:00.013895572,11056270537,Cancel,Buy,12.65,0.0,Limit,0,0.0,0.0,12.65,200.0
4336685,FR0000120172,2023-10-27 17:40:00.013895572,10989161673,Cancel,Buy,12.50,0.0,Limit,0,0.0,0.0,12.50,20.0
4336686,FR0000120172,2023-10-27 17:40:00.013895572,4731260105,Cancel,Buy,11.64,0.0,Limit,0,0.0,0.0,11.64,24.0


In [30]:
data = pd.read_parquet('FR0000120628_20231027_FO.parquet.gzip')
display(data)

,order_side,trade_price,trade_size
event_time_cet,,,
2023-10-27 09:00:00,Buy,27.610,477.0
2023-10-27 09:00:00,Buy,27.620,905.0
2023-10-27 09:00:00,Buy,27.625,419.0
2023-10-27 09:00:00,Buy,27.630,452.0
2023-10-27 09:00:00,Buy,27.650,402.0
...,...,...,...
2023-10-27 17:25:00,Sell,27.520,1880.0
2023-10-27 17:25:00,Sell,27.525,7307.0
2023-10-27 17:25:00,Sell,27.530,3564.0


In [33]:
display(pd.read_parquet('FR001400AJ45_20231027_LOB.parquet.gzip'))

,price,side,size
index,,,
2023-10-27 03:00:00,8.60,Buy,465.0
2023-10-27 03:00:00,13.45,Buy,8.0
2023-10-27 03:00:00,15.00,Buy,15.0
2023-10-27 03:00:00,17.00,Buy,300.0
2023-10-27 03:00:00,19.08,Buy,52.0
...,...,...,...
2023-10-27 17:40:00,40.00,Sell,400.0
2023-10-27 17:40:00,40.25,Sell,12.0
2023-10-27 17:40:00,40.75,Sell,124.0


In [4]:
mask = (FOB['order_event_type'] == 'New') | (FOB['order_event_type'] == 'Reload')
FOB['initial_index'] = FOB.index
FOB = pd.concat([FOB[mask], FOB[~mask]], ignore_index=True)

FOB['previous_price'] = FOB.groupby('order_id')['order_price'].shift(1)
FOB['previous_size'] = FOB.groupby('order_id')['order_size'].shift(1)

FOB = FOB.sort_values('initial_index').set_index('initial_index')
FOB.name = None

In [25]:
def resample_FOB_LOB(data, price: str, size: str, to_add: bool = True):
    resample_df = data.copy()

    resample_df = resample_df[['event_time_cet', 'order_side', 'order_type', 'time_in_force'] + [price, size]]

    if to_add == False:
        resample_df[size] *= -1

    resample_df.set_index('event_time_cet', inplace=True)
    resample_df = resample_df.groupby(['order_side', 'order_type', 'time_in_force', price]).resample('5min').sum()[size].to_frame()

    resample_df = resample_df[~resample_df.isna().any(axis=1)]
    resample_df = resample_df.reset_index().groupby(['event_time_cet', 'order_side', 'order_type', 'time_in_force', price], as_index=False).last()

    resample_df.columns = ['event_time_cet', 'side', 'order_type', 'time_in_force', 'price', 'size']

    return resample_df

In [33]:
data = FOB.copy()
data = data.loc[(data['time_in_force'] != '0')]
LOB_add = resample_FOB_LOB(data=data, price='order_price', size='order_size')
LOB_sub = resample_FOB_LOB(data=data, price='previous_price', size='previous_size', to_add=False)
resamp_FOB_LOB = pd.concat([LOB_add, LOB_sub], ignore_index=True).groupby(['event_time_cet', 'side', 'order_type', 'time_in_force', 'price'], as_index=False).sum()

In [34]:
display(resamp_FOB_LOB)

,event_time_cet,side,order_type,time_in_force,price,size
0,2023-10-27 07:55:00,Buy,Market,Valid for Uncrossing,0.00,2.0
1,2023-10-27 08:00:00,Buy,Market,Valid for Uncrossing,0.00,0.0
2,2023-10-27 08:05:00,Buy,Market,Valid for Uncrossing,0.00,0.0
3,2023-10-27 08:10:00,Buy,Market,Valid for Uncrossing,0.00,0.0
4,2023-10-27 08:15:00,Buy,Market,Valid for Uncrossing,0.00,0.0
...,...,...,...,...,...,...
1374,2023-10-27 17:35:00,Sell,Limit,Valid for Closing,24.00,-336.0
1375,2023-10-27 17:35:00,Sell,Limit,Valid for Closing,24.43,-250.0
1376,2023-10-27 17:35:00,Sell,Limit,Valid for Closing,24.88,-17.0
1377,2023-10-27 17:35:00,Sell,Limit,Valid for Closing,25.10,-4000.0


In [40]:
LOB = pd.DataFrame({'side':['Buy','Buy','Buy','Buy','Sell','Sell','Sell','Sell'],
                    'order_type':['Market','Market','Limit','Limit','Market','Market','Limit','Limit'],
                    'time_in_force':['Valid for Uncrossing',
                                     'Valid for Closing',
                                     'Valid for Uncrossing',
                                     'Valid for Closing',
                                     'Valid for Uncrossing',
                                     'Valid for Closing',
                                     'Valid for Uncrossing',
                                     'Valid for Closing'],
                    'size':[0,0,0,0,0,0,0,0]})

fin = pd.DataFrame()

for t, block in resamp_FOB_LOB.groupby('event_time_cet'):
    tmp = block[['side', 'order_type', 'time_in_force', 'size']]

    LOB = LOB.reset_index(drop=True)

    LOB = pd.concat([LOB, tmp], ignore_index=True).groupby(['side', 'order_type', 'time_in_force'], as_index=False).sum()
    #LOB = LOB[LOB['size'] != 0]
    LOB.index = pd.Index([t] * len(LOB))
    fin = pd.concat([fin, LOB])

In [42]:
display(fin)
display(fin.describe())

,side,order_type,time_in_force,size
2023-10-27 07:55:00,Buy,Limit,Valid for Closing,0.0
2023-10-27 07:55:00,Buy,Limit,Valid for Uncrossing,0.0
2023-10-27 07:55:00,Buy,Market,Valid for Closing,0.0
2023-10-27 07:55:00,Buy,Market,Valid for Uncrossing,2.0
2023-10-27 07:55:00,Sell,Limit,Valid for Closing,0.0
...,...,...,...,...
2023-10-27 17:35:00,Buy,Market,Valid for Uncrossing,0.0
2023-10-27 17:35:00,Sell,Limit,Valid for Closing,0.0
2023-10-27 17:35:00,Sell,Limit,Valid for Uncrossing,0.0
2023-10-27 17:35:00,Sell,Market,Valid for Closing,0.0


,size
count,520.000000
mean,4046.015385
std,38859.231158
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,663525.000000


In [17]:
t = resamp_FOB_LOB.groupby(['event_time_cet','side', 'order_type', 'time_in_force'], as_index=False).sum()
display(t[t['size'] != 0])

,event_time_cet,side,order_type,time_in_force,price,size
0,2023-10-27 07:55:00,Buy,Market,Valid for Uncrossing,0.000,2.0
8,2023-10-27 08:30:00,Sell,Limit,Valid for Uncrossing,16.030,843.0
13,2023-10-27 08:45:00,Buy,Limit,Valid for Uncrossing,700.455,4667.0
15,2023-10-27 08:45:00,Sell,Limit,Valid for Uncrossing,2304.525,19814.0
16,2023-10-27 08:45:00,Sell,Market,Valid for Uncrossing,0.000,187.0
17,2023-10-27 08:50:00,Buy,Limit,Valid for Uncrossing,700.455,-138.0
18,2023-10-27 08:50:00,Buy,Market,Valid for Uncrossing,0.000,414.0
19,2023-10-27 08:50:00,Sell,Limit,Valid for Uncrossing,2304.525,3166.0
20,2023-10-27 08:50:00,Sell,Market,Valid for Uncrossing,0.000,2462.0
21,2023-10-27 08:55:00,Buy,Limit,Valid for Uncrossing,1061.965,8023.0
